# 🌿 Plant Disease Detection — Advanced CNN Training
**Upgraded Stack:** TensorFlow/Keras · EfficientNetB0 · OpenCV
**Dataset:** Processed & Merged (PlantVillage + PlantDoc)

*Upgraded to use Explicit Train/Val/Test splits and the state-of-the-art EfficientNetB0 architecture.*
---

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report, confusion_matrix

print(f'TensorFlow : {tf.__version__}')
print(f'GPU(s)     : {tf.config.list_physical_devices("GPU")}')

TensorFlow : 2.21.0
GPU(s)     : []


In [2]:
# ── Configuration ──
# Upgraded to point to the new processed dataset splits
TRAIN_DIR    = Path('../dataset/processed/train')
VAL_DIR      = Path('../dataset/processed/val')
TEST_DIR     = Path('../dataset/processed/test')

MODEL_DIR    = Path('../model')
MODEL_PATH   = MODEL_DIR / 'plant_disease_model.keras'  # Modern Keras format
IMG_SIZE     = (224, 224)
BATCH_SIZE   = 16
EPOCHS       = 25
SEED         = 42

# Fix random seeds for reproducibility
np.random.seed(SEED)
tf.random.set_seed(SEED)

print('Configuration ready.')

Configuration ready.


## 1. Data Loading & Augmentation
*Upgraded: Uses explicit Train/Val/Test directories instead of arbitrary splits.*
*Note: EfficientNet natively handles input scaling, so rescale=1./255 is REMOVED to maximize accuracy.*

In [3]:
# ── ImageDataGenerators ──
# Training: strong augmentation to generalize to real-world data
train_datagen = ImageDataGenerator(
    rotation_range    = 40,
    width_shift_range = 0.2,
    height_shift_range= 0.2,
    shear_range       = 0.2,
    zoom_range        = 0.2,
    horizontal_flip   = True,
    vertical_flip     = True,
    fill_mode         = 'nearest'
)

# Validation & Test: NO augmentation
val_datagen  = ImageDataGenerator()
test_datagen = ImageDataGenerator()

print("Loading Training Data...")
train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size   = IMG_SIZE,
    batch_size    = BATCH_SIZE,
    class_mode    = 'categorical',
    shuffle       = True,
    seed          = SEED
)

print("Loading Validation Data...")
val_generator = val_datagen.flow_from_directory(
    VAL_DIR,
    target_size   = IMG_SIZE,
    batch_size    = BATCH_SIZE,
    class_mode    = 'categorical',
    shuffle       = False
)

print("Loading Test Data...")
test_generator = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size   = IMG_SIZE,
    batch_size    = BATCH_SIZE,
    class_mode    = 'categorical',
    shuffle       = False
)

CLASS_NAMES = list(train_generator.class_indices.keys())
NUM_CLASSES = len(CLASS_NAMES)

Loading Training Data...
Found 34845 images belonging to 41 classes.
Loading Validation Data...
Found 7393 images belonging to 41 classes.
Loading Test Data...
Found 7408 images belonging to 41 classes.


## 2. Model Architecture (EfficientNetB0)
*Upgraded from MobileNetV2 to EfficientNetB0. EfficientNet achieves significantly higher accuracy with fewer parameters.*

In [4]:
def build_model(num_classes):
    # EfficientNetB0 expects inputs in range [0, 255]
    base_model = EfficientNetB0(
        input_shape=(*IMG_SIZE, 3),
        include_top=False,
        weights='imagenet'
    )
    
    # Freeze base model initially
    base_model.trainable = False

    inputs = layers.Input(shape=(*IMG_SIZE, 3))
    x = base_model(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.4)(x)
    
    # Classification Head
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    
    model = models.Model(inputs, outputs, name='PlantDisease_EfficientNet')
    return model, base_model

model, base_model = build_model(NUM_CLASSES)
model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy', tf.keras.metrics.TopKCategoricalAccuracy(k=3, name='top3_acc')]
)
model.summary()

Model: "PlantDisease_EfficientNet"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─┼───┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─┼───┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─┼───┼───────────────┤
│ batch_normalization             │ (None, 1280)           │         5,120 │
│ (BatchNormalization)            │                        │               │
├─┼───┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─┼───┼───────────────┤
│ dense (Dense)                   │ (None, 41)             │        52,521 │
└─┴───┴───────────────┘

 Total params: 4,107,212 (15.67 MB)

 Trainable params: 55,081 (215.16 KB)

 Non-trainable params: 4,052,131 (15.46 MB)

## 3. Training Phase 1: Feature Extraction

In [5]:
# ── Callbacks ─────
checkpoint_cb = callbacks.ModelCheckpoint(str(MODEL_PATH), monitor='val_accuracy', save_best_only=True, verbose=1)
early_stop_cb = callbacks.EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True, verbose=1)
reduce_lr_cb  = callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1)

print('\n🚀 Starting Phase 1 Training (Feature Extraction)...')
history1 = model.fit(
    train_generator,
    epochs          = 10,
    validation_data = val_generator,
    callbacks       = [checkpoint_cb, early_stop_cb, reduce_lr_cb],
    verbose         = 1
)


🚀 Starting Phase 1 Training (Feature Extraction)...
Epoch 1/10
2178/2178 ━━━━━━━━━━━━━━━━━━━━ 0s 490ms/step - accuracy: 0.6975 - loss: 1.1118 - top3_acc: 0.8447
Epoch 1: val_accuracy improved from None to 0.90802, saving model to ..\model\plant_disease_model.keras

Epoch 1: finished saving model to ..\model\plant_disease_model.keras
2178/2178 ━━━━━━━━━━━━━━━━━━━━ 1257s 574ms/step - accuracy: 0.8061 - loss: 0.6574 - top3_acc: 0.9324 - val_accuracy: 0.9080 - val_loss: 0.3000 - val_top3_acc: 0.9847 - learning_rate: 0.0010
Epoch 2/10
2178/2178 ━━━━━━━━━━━━━━━━━━━━ 0s 471ms/step - accuracy: 0.8754 - loss: 0.4072 - top3_acc: 0.9749
Epoch 2: val_accuracy improved from 0.90802 to 0.92114, saving model to ..\model\plant_disease_model.keras

Epoch 2: finished saving model to ..\model\plant_disease_model.keras
2178/2178 ━━━━━━━━━━━━━━━━━━━━ 1215s 558ms/step - accuracy: 0.8801 - loss: 0.3937 - top3_acc: 0.9760 - val_accuracy: 0.9211 - val_loss: 0.2581 - val_top3_acc: 0.9869 - learning_rate: 0.001

## 4. Training Phase 2: Fine-Tuning
*Unfreezing the top layers of EfficientNet to adapt to plant-specific features.*

In [6]:
# Unfreeze the top 30 layers
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-4), # Lower LR for fine-tuning
    loss='categorical_crossentropy',
    metrics=['accuracy', tf.keras.metrics.TopKCategoricalAccuracy(k=3, name='top3_acc')]
)

print('\n🔥 Starting Phase 2 Fine-Tuning...')
history2 = model.fit(
    train_generator,
    epochs          = EPOCHS,
    initial_epoch   = len(history1.history['accuracy']),
    validation_data = val_generator,
    callbacks       = [checkpoint_cb, early_stop_cb, reduce_lr_cb],
    verbose         = 1
)


🔥 Starting Phase 2 Fine-Tuning...
Epoch 11/25
2178/2178 ━━━━━━━━━━━━━━━━━━━━ 0s 475ms/step - accuracy: 0.8694 - loss: 0.4607 - top3_acc: 0.9700
Epoch 11: val_accuracy improved from 0.93791 to 0.95658, saving model to ..\model\plant_disease_model.keras

Epoch 11: finished saving model to ..\model\plant_disease_model.keras
2178/2178 ━━━━━━━━━━━━━━━━━━━━ 1263s 575ms/step - accuracy: 0.9017 - loss: 0.3356 - top3_acc: 0.9821 - val_accuracy: 0.9566 - val_loss: 0.1395 - val_top3_acc: 0.9962 - learning_rate: 1.0000e-04
Epoch 12/25
2178/2178 ━━━━━━━━━━━━━━━━━━━━ 0s 688ms/step - accuracy: 0.9366 - loss: 0.2092 - top3_acc: 0.9930
Epoch 12: val_accuracy improved from 0.95658 to 0.96700, saving model to ..\model\plant_disease_model.keras

Epoch 12: finished saving model to ..\model\plant_disease_model.keras
2178/2178 ━━━━━━━━━━━━━━━━━━━━ 1706s 783ms/step - accuracy: 0.9396 - loss: 0.1983 - top3_acc: 0.9934 - val_accuracy: 0.9670 - val_loss: 0.1043 - val_top3_acc: 0.9969 - learning_rate: 1.0000e-04

## 5. Final Evaluation on Hold-out Test Set
*Upgraded: We now test the model on data it has NEVER seen during training or validation.*

In [11]:
# Load the absolute best weights saved by ModelCheckpoint
best_model = tf.keras.models.load_model(str(MODEL_PATH))

print('\n🧪 Evaluating on Test Data...')
test_loss, test_acc, test_top3 = best_model.evaluate(test_generator, verbose=1)
print(f'\nFinal Test Accuracy: {test_acc:.4f}')

# Detailed Classification Report
print('Generating detailed predictions...')
test_generator.reset()
y_pred_proba = best_model.predict(test_generator, verbose=1)
y_pred = np.argmax(y_pred_proba, axis=1)
y_true = test_generator.classes



🧪 Evaluating on Test Data...
463/463 ━━━━━━━━━━━━━━━━━━━━ 184s 385ms/step - accuracy: 0.9853 - loss: 0.0524 - top3_acc: 0.9991

Final Test Accuracy: 0.9853
Generating detailed predictions...
463/463 ━━━━━━━━━━━━━━━━━━━━ 183s 390ms/step


In [9]:
# Find only the classes that actually have images (ignores the 3 empty folders)
active_classes = np.unique(np.concatenate([y_true, y_pred]))
active_class_names = [CLASS_NAMES[i] for i in active_classes]

print('\nClassification Report:\n')
print(classification_report(y_true, y_pred, labels=active_classes, target_names=active_class_names, zero_division=0))



Classification Report:

                                                precision    recall  f1-score   support

                              Apple_Apple_scab       1.00      0.97      0.98       190
                               Apple_Black_rot       1.00      1.00      1.00       188
                        Apple_Cedar_apple_rust       1.00      1.00      1.00        84
                                 Apple_healthy       0.99      0.99      0.99       450
                             Blueberry_healthy       1.00      1.00      1.00       450
          Cherry_including_sour_Powdery_mildew       1.00      0.99      1.00       316
                 Cherry_including_sour_healthy       1.00      1.00      1.00       258
Corn_maize_Cercospora_leaf_spot_Gray_leaf_spot       0.88      0.97      0.93       154
                       Corn_maize_Common_rust_       1.00      0.99      0.99       358
               Corn_maize_Northern_Leaf_Blight       0.94      0.86      0.90       148
      

## 6. Save Class Names for Flask App

In [10]:
import json
with open('../model/class_names.json', 'w') as f:
    json.dump(CLASS_NAMES, f, indent=2)
print('✅ class_names.json saved successfully! Model is ready for deployment.')

✅ class_names.json saved successfully! Model is ready for deployment.
